# Therapeutic Optimization — Colab Runner

Choose `WORKFLOW_MODE` in the first code cell: `"basic"` preserves the original workflow; `"sophisticated"` runs the ordered all-site search.

**Basic:** T1_basic → UP1_basic → T2_basic → ESM2_basic → S1_basic → R1_basic → UB2_basic → R2_basic.

**Sophisticated:** T1_complex → UP1_complex → repeat (T2_complex → ESM2_complex → S1_complex) until 25 survivors, then R1_complex → UB2_complex → R2_complex.

All reusable logic lives in `src/therapeutic_optimization/`.


## Runtime, dependencies, imports

Use a **GPU runtime**. EUP uses ESM2-3B and this implementation intentionally fails rather than silently falling back to a very slow CPU path.


In [ ]:
WORKFLOW_MODE = "sophisticated"  # "basic" or "sophisticated"; rerun cells below after changing

from pathlib import Path
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/juliaevizza/therapeutic_optimization.git"
REPO_DIR = Path("/content/therapeutic_optimization")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Repository ready:", REPO_DIR)

# Run this after REPO_URL is configured and the repository has been cloned.
if REPO_DIR.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[eup]"], check=True)

    if shutil.which("git-lfs") is None:
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "git-lfs"], check=True)
    subprocess.run(["git", "lfs", "install"], check=True)

    if shutil.which("colabfold_batch") is None:
        print("colabfold_batch not found. Installing the current CUDA 12 ColabFold stack...")
        subprocess.run([
            sys.executable, "-m", "pip", "install", "-q",
            "colabfold[alphafold,openmm]", "jax[cuda12]", "openmm[cuda12]"
        ], check=True)

    subprocess.run([sys.executable, str(REPO_DIR / "scripts" / "check_environment.py")], check=False)


## 2. Choose where run outputs live

Set `SAVE_TO_DRIVE = True` if you want the run to survive Colab shutdown. The package code still runs from GitHub; only `storage/` outputs go into this run directory.


In [ ]:
SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_ROOT = Path("/content/drive/MyDrive/therapeutic_optimization_run")
else:
    RUN_ROOT = Path("/content/therapeutic_optimization_run")

RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("Run root:", RUN_ROOT)


## 3. Hyperparameters

Basic mode uses `MUTATION_MODE`: `single` changes one selected lysine at a time; `combinatorial` generates orders 1 through `MAX_COMBINATION_ORDER`.

Sophisticated mode mutates **every lysine whose UP1 probability is strictly above `UBI_THRESHOLD`**. It starts with all arginine (R), then pans histidine (H) through 1, 2, …, n sites. It then introduces glutamine (Q), glutamic acid (E), and cysteine (C), in that order, testing every new combination without repeats. The search stops at 25 structural survivors or after exhausting the five-amino-acid space.

`mean_residue_representation_cosine_similarity > 0.9995` is required before structural prediction. Pseudo-perplexity changes are recorded separately; set the optional percent-change limit to enforce a second ESM-2 gate (0 means no increase). ESM-2 similarity is a prescreen; the existing S1 structural thresholds still determine structural survival.


In [ ]:
UBI_THRESHOLD = 0.40

MUTATION_MODE = "single"            # "single" or "combinatorial"
REPLACEMENT_AAS = ("R",)            # e.g. ("A", "R", "Q")
MAX_COMBINATION_ORDER = 2
MAX_VARIANTS = 5000

# Sophisticated mode only. This explicit order ends at cysteine.
COMPLEX_REPLACEMENT_AAS = ("R", "H", "Q", "E", "C")
TARGET_SURVIVORS = 25
MIN_MEAN_RESIDUE_COSINE_SIMILARITY = 0.9995  # strict >, not >=
MAX_PSEUDO_PERPLEXITY_PERCENT_CHANGE = None # optional; 0.0 rejects any increase
MAX_COMPLEX_CANDIDATES = None              # optional limit on attempted candidates

# ESM2 WT-relative sequence/representation scoring
RUN_ESM2 = True
ESM2_MODEL = "facebook/esm2_t33_650M_UR50D"
ESM2_MASK_BATCH_SIZE = 4
ESM2_PERPLEXITY_WEIGHT = 0.70
ESM2_REPRESENTATION_WEIGHT = 0.30

# S1 structural-preservation heuristic gates
GLOBAL_CA_RMSD_MAX = 1.0
BINDER_START = 1                    # inclusive; use None for both to disable
BINDER_END = 16                     # PTAPPYDSLLVFDYEG in this construct
BINDER_CA_RMSD_MAX = 1.0            # binder shape after independent alignment
CORE_CA_RMSD_MAX = 1.0              # E3/core after excluding the binder
LOCAL_MEAN_CA_DISPLACEMENT_MAX = 1.5
MUTATION_CA_DISPLACEMENT_MAX = 2.0
CONTACT_CHANGE_FRACTION_MAX = 0.10
MIN_MEAN_PLDDT = 70.0


## 4. User input


In [ ]:
PROTEIN_ID = "my_protein"
WT_SEQUENCE = """PTAPPYDSLLVFDYEGGSGSGSGASRLNFGDDIPSALRIAKKKRWNSIEERRIHQESELHSYLSRLIAAERERELEECQRNHEGDEDDSHVRAQQACIEAKHDKYMADMDELFSQVDEKRKKRDIPDYLCGKISFELMREPCITPSGITYDRKDIEEHLQRVGHFDPVTRSPLTQEQLIPNLAMKEVIDAFISENGWVEDY"""


## 5. Build the workflow


In [ ]:
import sys
if 'therapeutic_optimization' in sys.modules:
    del sys.modules['therapeutic_optimization']
if "/content/therapeutic_optimization/src" not in sys.path:
    sys.path.insert(0, "/content/therapeutic_optimization/src")

from therapeutic_optimization.config import (
    ComplexSearchConfig,
    ESM2AnalysisConfig,
    MutationConfig,
    PredictorConfig,
    StructuralThresholds,
    WorkflowConfig,
)
from therapeutic_optimization.workflow import OptimizationWorkflow
from pathlib import Path

if WORKFLOW_MODE not in {"basic", "sophisticated"}:
    raise ValueError('WORKFLOW_MODE must be "basic" or "sophisticated".')
if WORKFLOW_MODE == "sophisticated" and not RUN_ESM2:
    raise ValueError("Sophisticated mode requires RUN_ESM2 = True.")

config = WorkflowConfig(
    mode=WORKFLOW_MODE,
    complex_search=ComplexSearchConfig(
        replacement_aas=COMPLEX_REPLACEMENT_AAS,
        target_survivors=TARGET_SURVIVORS,
        min_mean_residue_cosine_similarity=MIN_MEAN_RESIDUE_COSINE_SIMILARITY,
        max_pseudo_perplexity_percent_change=MAX_PSEUDO_PERPLEXITY_PERCENT_CHANGE,
        max_candidates=MAX_COMPLEX_CANDIDATES,
    ),
    mutation=MutationConfig(
        threshold=UBI_THRESHOLD,
        mode=MUTATION_MODE,
        replacement_aas=REPLACEMENT_AAS,
        max_combination_order=MAX_COMBINATION_ORDER,
        max_variants=MAX_VARIANTS,
    ),
    ubiquitination=PredictorConfig(
        name="eup",
        threshold=UBI_THRESHOLD,
        eup_repo_dir=Path("/content/external/EUP"),
        model_cache_dir=Path("/content/huggingface"),
    ),
    esm2=ESM2AnalysisConfig(
        model_name=ESM2_MODEL,
        mask_batch_size=ESM2_MASK_BATCH_SIZE,
        model_cache_dir=Path("/content/huggingface"),
        perplexity_weight=ESM2_PERPLEXITY_WEIGHT,
        representation_weight=ESM2_REPRESENTATION_WEIGHT,
    ),
    structural_thresholds=StructuralThresholds(
        global_ca_rmsd_max=GLOBAL_CA_RMSD_MAX,
        binder_start=BINDER_START,
        binder_end=BINDER_END,
        binder_ca_rmsd_max=BINDER_CA_RMSD_MAX,
        core_ca_rmsd_max=CORE_CA_RMSD_MAX,
        local_mean_ca_displacement_max=LOCAL_MEAN_CA_DISPLACEMENT_MAX,
        mutation_ca_displacement_max=MUTATION_CA_DISPLACEMENT_MAX,
        contact_change_fraction_max=CONTACT_CHANGE_FRACTION_MAX,
        min_mean_plddt=MIN_MEAN_PLDDT,
    ),
)

workflow = OptimizationWorkflow(RUN_ROOT, config)
STAGE_SUFFIX = workflow.stage_suffix
print("Selected stages:", STAGE_SUFFIX)
print("Output table example:", workflow.paths.table("T2_mutation_manifest.csv"))

## T1_basic / T1_complex — input → WT FASTA


In [ ]:
t1 = getattr(workflow, f"T1_{STAGE_SUFFIX}")(WT_SEQUENCE, PROTEIN_ID)
t1


## UP1_basic / UP1_complex — WT ubiquitination prediction


In [ ]:
up1 = getattr(workflow, f"UP1_{STAGE_SUFFIX}")()
display(up1)


## T2_basic / T2_complex — mutation generation / adaptive search

In sophisticated mode this cell runs the full adaptive T2_complex → ESM2_complex → S1_complex loop. The following ESM2/S1 cells display its saved results without repeating inference. Basic mode only generates FASTAs here.


In [ ]:
if WORKFLOW_MODE == "basic":
    t2_basic = workflow.T2_basic(up1)
    t2 = t2_basic
else:
    t2_complex = workflow.T2_complex(up1, predict_structures=True)
    t2 = t2_complex
    import json
    print(workflow.paths.table("T2_search_summary.json").read_text())
print(f"T2_{STAGE_SUFFIX}: {len(t2)} candidate(s) generated and recorded.")
display(t2)


## ESM2_basic / ESM2_complex — WT-relative plausibility + representation screen


In [ ]:
esm2 = None
if RUN_ESM2:
    if WORKFLOW_MODE == "basic":
        try:
            esm2_basic = workflow.ESM2_basic(t2)
        finally:
            workflow.esm2_scorer.release()
        esm2 = esm2_basic
    else:
        esm2_complex = workflow.ESM2_complex(t2)
        esm2 = esm2_complex
    display(esm2)


## S1_basic / S1_complex — structural preservation


In [ ]:
if WORKFLOW_MODE == "basic":
    s1_metrics_basic, s1_conserved_basic = workflow.S1_basic(t2, predict_structures=True)
    s1_metrics, s1_conserved = s1_metrics_basic, s1_conserved_basic
else:
    s1_metrics_complex, s1_conserved_complex = workflow.S1_complex(t2)
    s1_metrics, s1_conserved = s1_metrics_complex, s1_conserved_complex
print(f"S1_{STAGE_SUFFIX}: {len(s1_conserved)} structurally conserved / {len(t2)} attempted")
display(s1_metrics)
display(s1_conserved)


## R1_basic / R1_complex — record screen outcomes


In [ ]:
r1 = getattr(workflow, f"R1_{STAGE_SUFFIX}")(t2, s1_metrics)
display(r1)


## UB2_basic / UB2_complex — ubiquitination prediction for survivors


In [ ]:
ub2 = getattr(workflow, f"UB2_{STAGE_SUFFIX}")(s1_conserved)
display(ub2)


## R2_basic / R2_complex — final ranking


In [ ]:
r2_all, optimized, needs_more = getattr(workflow, f"R2_{STAGE_SUFFIX}")(
    up1,
    ub2,
    s1_conserved,
    esm2_results=esm2,
    use_saved_esm2=RUN_ESM2,
)

print("OPTIMIZED")
display(optimized)

print("NEEDS FURTHER OPTIMIZATION")
display(needs_more)

print("ALL RANKED")
display(r2_all)


## 6. Inspect outputs


In [ ]:
print("Tables:")
for path in sorted((RUN_ROOT / "storage" / "tables").glob("*.csv")):
    print(" -", path)

print("Mutant FASTAs:", len(list((RUN_ROOT / "storage" / "mutants" / "fastas").glob("*.fasta"))))
print("WT structure directory:", RUN_ROOT / "storage" / "structures" / "wt")
print("Mutant structure directory:", RUN_ROOT / "storage" / "structures" / "mutants")
